# Экспертная валидация извлечения экспериментов из научных статей (Google Colab)

[![Открыть в Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/top-papers/scientific-extraction-harness/blob/main/notebooks/scientific_experiment_review.ipynb)

Или ссылка для открытия в Colab без бейджа:
`https://colab.research.google.com/github/top-papers/scientific-extraction-harness/blob/main/notebooks/scientific_experiment_review.ipynb`

Ноутбук заменяет интерактивную работу эксперта с OpenCode-агентом: **загрузить разрешённую статью → получить кандидатные EvidenceRecord → проверить каждое значение в HTML-форме → скачать или отправить только одобренные записи в Hugging Face**.

Контракт записи следует `docs/data-contract.md` проекта `scientific-extraction-harness`: `document_id`, страница, modality, locator, field, value, unit, confidence и статус. Модель не имеет права превращать отсутствие свидетельства в значение: сомнительные элементы помечаются `candidate`/`rejected` и требуют решения эксперта.

> Перед запуском добавьте секреты Colab: `HF_TOKEN` с правом создания PR в датасете и, если нужен доступ к приватным материалам GitHub, `top_papers_token` (Notebook access включать не обязательно). Токены не печатаются и не записываются в HTML/JSON. `top_papers_token` используется только как GitHub Bearer-токен; публикация в Hugging Face использует `HF_TOKEN`.

Исходные PDF не публикуются. В dataset отправляются только обезличенные структурированные записи и provenance, если у эксперта есть право на их распространение.

In [ ]:
%pip install -q -U pypdf requests
print('Dependencies ready: pypdf, requests')

In [ ]:
import base64, datetime as dt, hashlib, html, io, json, os, re, secrets, textwrap, time, urllib.parse, zipfile
from pathlib import Path

DATASET_REPO = 'top-papers/scientific-experiment-evidence'
HF_TOKEN_SECRET = 'HF_TOKEN'
GITHUB_TOKEN_SECRET = 'top_papers_token'
GITHUB_REPO = 'top-papers/scientific-extraction-harness'
MODEL = 'Qwen/Qwen2.5-72B-Instruct'
HF_INFERENCE_URL = 'https://api-inference.huggingface.co/models/' + MODEL
MAX_UPLOAD_MB = 25
MAX_TEXT_CHARS = 50000
EXPERT_NAME = ''
RUN_ID = 'run-' + dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + secrets.token_hex(3)

def secret(name):
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = os.environ.get(name)
    return (value or '').strip()

HF_TOKEN = secret(HF_TOKEN_SECRET)
TOP_PAPERS_TOKEN = secret(GITHUB_TOKEN_SECRET)
print('run:', RUN_ID)
print('HF_TOKEN:', 'configured' if HF_TOKEN else 'not configured')
print('top_papers_token:', 'configured' if TOP_PAPERS_TOKEN else 'not configured (needed only for private GitHub sources)')
print('model:', MODEL)

## 1. Загрузка статьи

Выберите PDF/TXT/HTML/DOCX/RTF через стандартный Colab upload. Можно также указать разрешённую локальную ссылку или GitHub raw URL. Для GitHub URL заголовок Authorization формируется из секрета `top_papers_token`, но его значение никогда не попадает в вывод. Сканированные PDF без текстового слоя нужно предварительно OCR-обработать: этот минимальный ноутбук не выдумывает данные по картинке.

In [ ]:
ALLOWED_EXT = {'.pdf', '.txt', '.md', '.html', '.htm', '.docx', '.rtf'}
UPLOAD_DIR = Path('/content/scientific_uploads')
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_PATH = ''
SOURCE_URL = ''

def safe_filename(name):
    name = re.sub(r'[^A-Za-zА-Яа-яЁё0-9._() -]+', '_', Path(name).name)[:150]
    if Path(name).suffix.lower() not in ALLOWED_EXT: raise ValueError('Unsupported file type')
    return name

def download_source(url):
    import requests
    headers = {'User-Agent': 'scientific-extraction-colab/1.0'}
    if 'github.com' in urllib.parse.urlparse(url).netloc and TOP_PAPERS_TOKEN:
        headers['Authorization'] = 'Bearer ' + TOP_PAPERS_TOKEN
    r = requests.get(url, headers=headers, timeout=90)
    r.raise_for_status()
    data = r.content
    if len(data) > MAX_UPLOAD_MB * 1024 * 1024: raise ValueError('File exceeds size limit')
    name = safe_filename(Path(urllib.parse.urlparse(url).path).name or 'article.pdf')
    path = UPLOAD_DIR / name
    path.write_bytes(data)
    return path

files = []
if LOCAL_PATH:
    files = [Path(LOCAL_PATH)]
elif SOURCE_URL:
    files = [download_source(SOURCE_URL)]
else:
    from google.colab import files as colab_files
    uploaded = colab_files.upload()
    for name, data in uploaded.items():
        try:
            clean = safe_filename(name)
            if len(data) > MAX_UPLOAD_MB * 1024 * 1024: raise ValueError('file too large')
            path = UPLOAD_DIR / clean; path.write_bytes(data); files.append(path)
        except Exception as exc: print('Skipped', name, ':', exc)
print('Documents:', [p.name for p in files])

In [ ]:
def extract_text(path):
    data = path.read_bytes(); ext = path.suffix.lower()
    if ext == '.pdf':
        from pypdf import PdfReader
        reader = PdfReader(io.BytesIO(data))
        pages = [(p.extract_text() or '').strip() for p in reader.pages[:300]]
        return '\n'.join(f'[[PAGE {i+1}]]\n{x}' for i, x in enumerate(pages))
    if ext == '.docx':
        with zipfile.ZipFile(io.BytesIO(data)) as z: xml = z.read('word/document.xml').decode('utf-8', 'replace')
        return re.sub(r'<[^>]+>', ' ', xml.replace('</w:p>', '\n'))
    text = data.decode('utf-8', 'replace')
    if ext in {'.html', '.htm'}: text = re.sub(r'(?is)<(script|style)[^>]*>.*?</\1>', ' ', text); text = re.sub(r'<[^>]+>', ' ', text)
    if ext == '.rtf': text = re.sub(r'\\[a-z]+-?\d* ?', ' ', text).replace('\\par', '\n')
    return html.unescape(text)

def normalize(s):
    lines = [re.sub(r'[ \t]+', ' ', x).strip() for x in s.replace('\xa0',' ').splitlines()]
    return '\n'.join(x for i, x in enumerate(lines) if x or (i and lines[i-1]))[:MAX_TEXT_CHARS]

DOCS = []
for path in files:
    raw = path.read_bytes(); text = normalize(extract_text(path))
    doc_id = 'doc-' + hashlib.sha256(raw).hexdigest()[:16]
    DOCS.append({'document_id': doc_id, 'filename': path.name, 'sha256': hashlib.sha256(raw).hexdigest(), 'text': text})
    print(path.name, '|', doc_id, '| chars:', len(text), '| pages:', text.count('[[PAGE '))
    if len(text.strip()) < 200: print('WARNING: almost no text; use OCR before extraction')

## 2. Извлечение кандидатных EvidenceRecord

Модель получает только текст и инструкцию вернуть JSON-массив. Схема намеренно требует page/modality/locator и дословное значение. Поля без надёжного свидетельства должны иметь `value: null` и `status: candidate`; они не попадут в одобренную выгрузку без решения эксперта.

In [ ]:
EXTRACTION_SCHEMA = {
  'document_id':'stable id', 'page':1, 'modality':'text|table|figure|scan',
  'locator':'page paragraph/table/figure locator', 'field':'normalized variable',
  'value':'verbatim value or null', 'unit':'source unit or null', 'confidence':0.0,
  'status':'candidate', 'evidence_quote':'short verbatim quote', 'reviewer_note':''
}
PROMPT = '''You extract experimental data from a scientific article. Return ONLY a JSON array of records.
Never infer a value not supported by the text. Preserve values and units verbatim. For every record provide page (1-based; use 1 only if no page markers), modality text/table/figure/scan, and a precise locator. Include experimental conditions, sample sizes, interventions, outcomes, metrics, uncertainty/error bars, and comparison groups when explicitly reported. If ambiguous, set value to null and confidence <= 0.4. Every record must include evidence_quote.
JSON object shape: ''' + json.dumps(EXTRACTION_SCHEMA, ensure_ascii=False)

def call_model(text):
    import requests
    if not HF_TOKEN: raise RuntimeError('Configure HF_TOKEN in Colab Secrets for model extraction')
    payload = {'inputs': PROMPT + '\nARTICLE:\n' + text, 'parameters': {'max_new_tokens': 5000, 'temperature': 0.0, 'return_full_text': False}}
    r = requests.post(HF_INFERENCE_URL, headers={'Authorization':'Bearer '+HF_TOKEN}, json=payload, timeout=240)
    r.raise_for_status(); body = r.json()
    generated = body[0].get('generated_text','') if isinstance(body,list) else body.get('generated_text','')
    match = re.search(r'\[.*\]', generated, re.S)
    if not match: raise ValueError('Model did not return a JSON array')
    return json.loads(match.group(0))

def normalize_candidates(doc, candidates):
    out=[]
    for x in candidates:
        if not isinstance(x, dict): continue
        x = dict(x); x['document_id']=doc['document_id']; x['status']='candidate'
        x['page'] = int(x.get('page') or 1); x['modality'] = x.get('modality') if x.get('modality') in {'text','table','figure','scan'} else 'text'
        x['confidence'] = max(0.0, min(1.0, float(x.get('confidence') or 0.0)))
        x.setdefault('unit', None); x.setdefault('evidence_quote',''); x.setdefault('reviewer_note','')
        out.append(x)
    return out

CANDIDATES=[]
for doc in DOCS:
    try:
        CANDIDATES += normalize_candidates(doc, call_model(doc['text']))
        print('Extracted candidates:', len(CANDIDATES))
    except Exception as exc: print('Extraction failed:', type(exc).__name__, str(exc)[:300])

## 3. HTML-форма экспертной проверки

В форме эксперт должен выставить каждому полю `approve`, `reject` или `fix`, при необходимости исправить значение, добавить заметку и подтвердить проверку. Кнопка `Download review JSON` работает офлайн. Публикация создаёт PR в Hugging Face только после явного подтверждения; в dataset отправляются записи со статусом `approved` и provenance, а не исходная статья.

In [ ]:
FORM = r'''<!doctype html><meta charset=utf-8><title>Scientific evidence review</title>
<style>body{font:15px system-ui;max-width:1100px;margin:2em auto;color:#182230} .rec{border:1px solid #ccd5df;border-radius:10px;padding:14px;margin:12px 0} .quote{background:#f5f7fa;padding:8px;white-space:pre-wrap} label{margin-right:10px} input,textarea{padding:6px;margin:4px;max-width:100%} textarea{width:95%;min-height:50px} button{padding:9px 14px;margin:8px 4px;cursor:pointer}.bad{color:#a00}.ok{color:#176b36}</style>
<h1>Экспертная проверка EvidenceRecord</h1><p>Проверьте страницу, modality, locator и цитату по статье. Автоматическое одобрение запрещено.</p>
<label>Эксперт <input id=expert></label><label>Оценка 0–10 <input id=score type=number min=0 max=10></label><br><textarea id=comments placeholder='Общий комментарий, ошибки и неоднозначности'></textarea><div id=records></div><label><input id=consent type=checkbox> Я лично проверил(а) записи и provenance</label><br><button id=download>Download review JSON</button><button id=publish>Publish approved to Hugging Face</button><p id=status></p>
<script>const DATA=__PAYLOAD__; const esc=x=>String(x??'').replace(/[&<>"']/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c])); const state=DATA.records.map(()=>({}));
function render(){document.querySelector('#records').innerHTML=DATA.records.map((r,i)=>'<div class=rec><b>#'+(i+1)+' · '+esc(r.field||'field')+'</b><p><b>Candidate:</b> '+esc(r.value)+' '+esc(r.unit||'')+' · confidence '+esc(r.confidence)+'</p><p><b>Source:</b> page '+esc(r.page)+' · '+esc(r.modality)+' · '+esc(r.locator)+'</p><div class=quote>'+esc(r.evidence_quote)+'</div><label>Status <select data-i='+i+' class=status><option value=unchecked>unchecked</option><option value=approved>approve</option><option value=rejected>reject</option><option value=fixed>fix</option></select></label><input data-i='+i+' class=fix placeholder=&quot;Corrected value&quot;><br><input data-i='+i+' class=note placeholder=&quot;Reviewer note&quot; style=&quot;width:90%&quot;></div>').join('')} render();
function build(){if(!document.querySelector('#consent').checked) throw Error('Confirm expert review first'); const rs=DATA.records.map((r,i)=>{const s=document.querySelector('.status[data-i="'+i+'"]'); const f=document.querySelector('.fix[data-i="'+i+'"]'); const n=document.querySelector('.note[data-i="'+i+'"]'); return {...r,status:s.value,value:s.value==='fixed'?f.value:r.value,reviewer_note:n.value}}); if(rs.some(r=>r.status==='unchecked')) throw Error('Review every candidate'); const score=Number(document.querySelector('#score').value); if(!(score>=0&&score<=10)) throw Error('Set score 0–10'); return {schema_version:'0',kind:'expert_review',review_id:DATA.review_id,run_id:DATA.run_id,created_at:new Date().toISOString(),expert:document.querySelector('#expert').value.trim()||null,score,comments:document.querySelector('#comments').value.trim(),expert_approved:true,document:DATA.document,records:rs}}
function safeReview(x){const s=JSON.stringify(x); if(/hf_[A-Za-z0-9]{10,}/.test(s)||/top_papers_token|HF_TOKEN|api[_-]?key/i.test(s)) throw Error('Secret-like content detected'); return x} function dl(x){const a=document.createElement('a');a.href=URL.createObjectURL(new Blob([JSON.stringify(x,null,2)],{type:'application/json'}));a.download=x.review_id+'.json';a.click()}
const ST=document.querySelector('#status');function say(t,k){ST.textContent=t;ST.className=k||''}function onDownload(){try{const x=safeReview(build());dl(x);say('Review downloaded','ok')}catch(err){say(err.message,'bad')}}function onPublish(){try{const x=safeReview(build());window.reviewForColab=x;say('Review is ready. Return to the notebook and run the publication cell.','ok')}catch(err){say(err.message,'bad')}}document.addEventListener('click',ev=>{const id=ev.target&&ev.target.id;if(id==='download'){onDownload()}else if(id==='publish'){onPublish()}});</script>'''

def render_form(doc, records):
    payload={'run_id':RUN_ID,'review_id':'review-'+secrets.token_hex(6),'document':{'document_id':doc['document_id'],'filename':doc['filename'],'sha256':doc['sha256']},'records':records}
    return FORM.replace('__PAYLOAD__', json.dumps(payload, ensure_ascii=False).replace('</','<\/'))

FORM_PATH = Path('/content/expert_review.html')

if not DOCS:
    print('Нет загруженных документов. Выполните ячейки раздела 1 (загрузка статьи) и раздела 2 (извлечение), затем запустите эту ячейку снова.')
elif not CANDIDATES:
    print('Кандидаты не извлечены (CANDIDATES пуст). Проверьте, что HF_TOKEN задан и ячейка извлечения отработала без ошибок.')
    print('Публиковать нечего: форма требует хотя бы одну кандидатную запись.')
else:
    FORM_PATH.write_text(render_form(DOCS[0], CANDIDATES), encoding='utf-8')
    print('Форма сохранена:', FORM_PATH, '| записей:', len(CANDIDATES))
    try:
        from IPython.display import HTML, display
        display(HTML(FORM_PATH.read_text(encoding='utf-8')))
    except Exception as exc:
        print('Не удалось показать форму inline:', type(exc).__name__, exc)
    print('Если форма не отрисовалась ниже, скачайте файл и откройте его в браузере:')
    try:
        from google.colab import files as colab_files
        colab_files.download(str(FORM_PATH))
    except Exception:
        print('  путь к файлу:', FORM_PATH)

In [ ]:
def validate_review(review):
    if review.get('kind') != 'expert_review' or review.get('expert_approved') is not True: return 'missing explicit expert approval'
    if not isinstance(review.get('score'), (int,float)) or not 0 <= review['score'] <= 10: return 'score must be 0..10'
    if not isinstance(review.get('records'), list) or not review['records']: return 'no records'
    required={'document_id','page','modality','locator','field','value','unit','confidence','status','evidence_quote'}
    for i,r in enumerate(review['records']):
        if not required.issubset(r): return f'missing fields in record {i}'
        if r['status'] not in {'approved','rejected','candidate','fixed'}: return f'bad status in record {i}'
        if not isinstance(r['page'], int) or r['page'] < 1: return f'bad page in record {i}'
        if not 0 <= float(r['confidence']) <= 1: return f'bad confidence in record {i}'
        if re.search(r'hf_[A-Za-z0-9]{10,}|top_papers_token|HF_TOKEN|api[_-]?key', json.dumps(r), re.I): return 'secret-like value found'
    return None

def get_review_from_colab():
    rv = globals().get('reviewForColab')
    if rv is not None: return rv
    candidates = sorted(Path('/content').glob('review-*.json'))
    if not candidates: raise FileNotFoundError('Run Publish in the HTML form or upload downloaded review JSON to /content')
    return json.loads(candidates[-1].read_text(encoding='utf-8'))

def publish_review(review):
    import requests
    problem=validate_review(review)
    if problem: raise ValueError(problem)
    if not HF_TOKEN: raise RuntimeError('Configure HF_TOKEN in Colab Secrets')
    approved=[r for r in review['records'] if r['status'] in {'approved','fixed'}]
    if not approved: raise ValueError('No approved records to publish')
    lines=''.join(json.dumps(r, ensure_ascii=False)+'\n' for r in approved)
    path='evidence_records.jsonl'; payload=[{'key':'header','value':{'summary':'Expert review '+review['review_id']}},{'key':'file','value':{'path':path,'encoding':'base64','content':base64.b64encode(lines.encode()).decode()}}]
    url=f'https://huggingface.co/api/datasets/{DATASET_REPO}/commit/main?create_pr=1'
    r=requests.post(url,headers={'Authorization':'Bearer '+HF_TOKEN,'Content-Type':'application/x-ndjson'},data=''.join(json.dumps(x)+'\n' for x in payload),timeout=120)
    HF_TOKEN=''
    if not r.ok: raise RuntimeError(f'Hugging Face HTTP {r.status_code}: {r.text[:300]}')
    return r.json()

# After expert review, execute: result = publish_review(get_review_from_colab()); print('PR created:', result.get('url', result.get('commitUrl','see response')))

## Чеклист перед публикацией

- Эксперт сверил **каждое** значение с указанной страницей/таблицей/рисунком и исправил неоднозначности.
- В опубликованных строках нет исходного PDF, секретов, имён/контактов и необезличенных чувствительных данных.
- Лицензия статьи допускает распространение структурированных результатов.
- `HF_TOKEN` имеет право записи, а публикация создаёт PR (`create_pr=1`); токен после запроса очищается из переменной.
- `top_papers_token` нужен только для доступа к приватному GitHub-источнику и не является токеном Hugging Face.